# Elastic Net Regression (L1 + L2)

Combines Ridge (L2) and Lasso (L1) - handles correlated features AND provides feature selection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score

## 1. Load & Prepare Data

In [ ]:
X, y = load_diabetes(return_X_y=True)
feature_names = load_diabetes().feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Data: {X.shape}')

## 2. Compare Ridge vs Lasso vs Elastic Net

In [ ]:
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=1.0)
enet = ElasticNet(alpha=1.0, l1_ratio=0.5)

for name, model in [('Ridge', ridge), ('Lasso', lasso), ('ElasticNet', enet)]:
    model.fit(X_train_scaled, y_train)

print('Comparison (alpha=1.0):')
print('-' * 60)
for name, model in [('Ridge', ridge), ('Lasso', lasso), ('ElasticNet', enet)]:
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    n_feat = np.sum(model.coef_ != 0)
    print(f'{name:12s} RMSE: {rmse:6.2f}  R2: {r2:7.4f}  Features: {n_feat}/10')

## 3. Coefficient Comparison

In [ ]:
df = pd.DataFrame({
    'Feature': feature_names,
    'Ridge': np.round(ridge.coef_, 2),
    'Lasso': np.round(lasso.coef_, 2),
    'ElasticNet': np.round(enet.coef_, 2)
})
print(df.to_string(index=False))

## 4. Effect of l1_ratio Parameter (0=Ridge, 1=Lasso)

In [ ]:
results = []
for ratio in [0.1, 0.3, 0.5, 0.7, 0.9]:
    model = ElasticNet(alpha=1.0, l1_ratio=ratio)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    n_feat = np.sum(model.coef_ != 0)
    results.append({'l1_ratio': ratio, 'RMSE': rmse, 'Features': n_feat})

print(pd.DataFrame(results).to_string(index=False))

## 5. Visualize l1_ratio Effect

In [ ]:
results_df = pd.DataFrame(results)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(results_df['l1_ratio'], results_df['RMSE'], 'o-', color='blue')
ax1.set_xlabel('l1_ratio (0=Ridge, 1=Lasso)')
ax1.set_ylabel('RMSE')
ax1.set_title('Performance vs l1_ratio')
ax1.grid(True, alpha=0.3)

ax2.plot(results_df['l1_ratio'], results_df['Features'], 'o-', color='green')
ax2.set_xlabel('l1_ratio (0=Ridge, 1=Lasso)')
ax2.set_ylabel('Features Kept')
ax2.set_title('Sparsity vs l1_ratio')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Auto-tune Both Parameters with ElasticNetCV

In [ ]:
enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], cv=5, random_state=2)
enet_cv.fit(X_train_scaled, y_train)

y_pred_cv = enet_cv.predict(X_test_scaled)
rmse_cv = np.sqrt(mean_squared_error(y_test, y_pred_cv))
n_selected = np.sum(enet_cv.coef_ != 0)

print(f'Best alpha: {enet_cv.alpha_:.4f}')
print(f'Best l1_ratio: {enet_cv.l1_ratio_:.2f}')
print(f'RMSE: {rmse_cv:.2f}, Features: {n_selected}/10')